# Lab 19 — Production-Grade GraphRAG vs Flat RAG

**Submission notebook · AICB-K34 Track 3**

Notebook này là entrypoint thực thi end-to-end cho bài Lab 19. Pipeline được khóa theo policy:

- **FIRST_5000_ROWS_ONLY** — chỉ sử dụng đúng 5.000 dòng đầu của HackerNoon (index 0–4999), giữ nguyên thứ tự, không random sample.
- Golden benchmark canonical: `data/graphrag_golden_50_first5000.csv` (50 câu: factoid, multi-hop, cross-doc).
- Neo4j ingestion dùng `UNWIND` batch + provenance bắt buộc.
- Groq có timeout, pacing, `Retry-After`/exponential backoff và checkpoint để phù hợp free-tier rate limit.
- Bonus: Near-Dedup SimHash/LSH, Community Reports/Global Search, Self-Correction hop2 → hop3 → vector fallback.

> CI đặt `LAB_RUN_MODE=smoke` hoặc `full`. Trên Colab/local có thể đặt biến này trước khi chạy.

In [ ]:
import os
from pathlib import Path
import pandas as pd

from lab19_models import DEFAULT_GROQ_MODEL
from lab19_utils import load_golden_dataset, validate_golden_dataset

os.environ.setdefault("GROQ_FAST_MODEL", DEFAULT_GROQ_MODEL)
RUN_MODE = os.getenv("LAB_RUN_MODE", "smoke").strip().lower()
assert RUN_MODE in {"smoke", "full"}

print("Run mode:", RUN_MODE)
print("Source policy: FIRST_5000_ROWS_ONLY")
print("Groq model:", os.environ["GROQ_FAST_MODEL"] )

## 1. Golden dataset validation

Không tạo lại golden answer trong notebook. Dataset do đề/lớp cung cấp được dùng trực tiếp và được validate trước khi benchmark.

In [ ]:
GOLDEN_PATH = Path("data/graphrag_golden_50_first5000.csv")
golden_df = load_golden_dataset(GOLDEN_PATH)
golden_report = validate_golden_dataset(golden_df)
display(pd.DataFrame([golden_report]))
display(golden_df.groupby("group").size().rename("questions").to_frame())
assert golden_report["rows"] == 50

## 2. Architecture & rubric mapping

`run_lab()` thực hiện tuần tự:

1. Stream **first 5,000 rows** → exact dedup → Near-Dedup SimHash/LSH → chunking.
2. Conservative coreference (chỉ gọi LLM khi chunk có trigger) → NER/RE strict schema.
3. Entity Resolution: manual aliases + FAISS ANN + lexical/type guard + DSU audit.
4. Neo4j constraints/indexes → `UNWIND` bulk node/edge ingestion → zero-missing-provenance assertion.
5. Flat RAG FAISS và Hybrid GraphRAG (entity seed matching + BFS + super-node mitigation).
6. Self-Correction: hop2 → hop3 → vector fallback.
7. Golden benchmark + LLM-as-a-Judge: comprehensiveness, faithfulness, multi-hop reasoning, latency, token usage.
8. Community detection + Community Reports/Global Search.
9. Export rubric evidence và reports.

Các relation của Knowledge Graph vẫn giữ đúng allowlist của đề:
`ACQUIRED, DEVELOPED, INVESTED_IN, FOUNDED, WORKED_AT, PARTNERED_WITH, USES, LEADS`.

In [ ]:
from lab19_runtime import run_lab

manifest = run_lab(RUN_MODE)
display(pd.DataFrame([manifest]))

## 3. Rubric evidence

Các file dưới đây được sinh từ chính run hiện tại, không hard-code benchmark result.

In [ ]:
OUTPUTS = Path("outputs")
expected = [
    "graphrag_eval_results.csv",
    "graphrag_vs_flatrag_summary.csv",
    "entity_resolution_audit.csv",
    "supernode_diagnostics.csv",
    "community_reports.csv",
    "bonus_metrics.csv",
]
for name in expected:
    path = OUTPUTS / name
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")
    assert path.exists(), path

In [ ]:
eval_df = pd.read_csv(OUTPUTS / "graphrag_eval_results.csv")
summary_df = pd.read_csv(OUTPUTS / "graphrag_vs_flatrag_summary.csv")
display(summary_df)
display(eval_df[[
    "id", "group",
    "flat_comprehensiveness", "graph_comprehensiveness",
    "flat_faithfulness", "graph_faithfulness",
    "flat_multi_hop_reasoning", "graph_multi_hop_reasoning",
    "flat_latency_s", "graph_latency_s",
    "graph_route", "graph_edges", "graph_supernode_events"
]].head(10))

## 4. Failure-mode evidence

- **Entity Resolution audit:** `MERGE_MANUAL`, `MERGE_VECTOR`, `REJECT_GUARD`.
- **Super-node:** degree > 100 ⇒ tối đa 50 edge mới nhất, đồng thời có `GLOBAL_EDGE_CAP`.
- **Provenance:** mỗi edge phải có `source_chunk_id`, `published_date`, `evidence`.

In [ ]:
audit_df = pd.read_csv(OUTPUTS / "entity_resolution_audit.csv")
supernode_df = pd.read_csv(OUTPUTS / "supernode_diagnostics.csv")
display(audit_df.head(20))
display(supernode_df.head(15))

graph_checks_path = OUTPUTS / "graph_checks.json"
if graph_checks_path.exists():
    import json
    graph_checks = json.loads(graph_checks_path.read_text(encoding="utf-8"))
    display(pd.DataFrame([graph_checks]))
    assert graph_checks["invalid_provenance_edges"] == 0

## 5. Bonus evidence

### Bonus A — Near-Dedup
SimHash64 + LSH buckets; tránh pairwise cosine O(N²), có audit cặp drop/keep.

### Bonus B — Global Search via Community Reports
NetworkX Louvain → ghi `community_id` về Neo4j → tạo community reports → semantic global-search demo.

### Bonus C — Self-Correction Graph Retrieval
GraphRAG bắt đầu hop2; thiếu context thì hop3; vẫn thiếu thì hybrid vector fallback, có stop condition và route diagnostics.

In [ ]:
bonus_df = pd.read_csv(OUTPUTS / "bonus_metrics.csv")
community_df = pd.read_csv(OUTPUTS / "community_reports.csv")
display(bonus_df)
display(community_df.head(10))

## 6. Submission artifacts

Sau full run, repo/artifact cần có:

- executed notebook này;
- `outputs/graphrag_eval_results.csv`;
- `outputs/graphrag_vs_flatrag_summary.csv`;
- audit/diagnostic/bonus CSV;
- `reports/lab_report.md`;
- `reports/technical_defense.md`;
- `reports/failure_analysis.md`;
- `reports/reflection_ChuNguyenTuanAnh.md`.

Không hard-code API key/password trong notebook hoặc repo.